# 손상 차량 부품 분류 — EfficientNet-B3 로컬 학습

이 노트북은 손상 bbox와 차량 `part` bbox를 연결해 손상 위치를 10개 부품 그룹(앞범퍼, 앞펜더, 전조등, 뒷범퍼, 뒷펜더, 후미등, 옆면, 사이드미러, 스텝, 타이어/휠)으로 분류합니다. 타이어와 휠은 데이터에 별도 라벨이 없어 하나로 합칩니다.

## 실행 전
1. VS Code에서 저장소 루트를 열고 로컬 Python 커널을 선택한 뒤 셀을 순서대로 실행합니다.
2. 설정 셀의 `DATA_ROOT`가 `train/part/.../01.데이터` 폴더인지 확인합니다.
3. 기본 설정은 10개 클래스별 train 200개·validation 50개 crop(최대 2,000/500장), ImageNet 사전학습 가중치, 224px 입력, batch 4, 최대 5 epoch입니다. 검증 Macro F1이 2 epoch 연속 개선되지 않으면 일찍 멈춥니다.
4. 첫 실행 시 사전학습 가중치(약 50MB)를 다운로드합니다. train/validation 라벨 JSON 표본은 최대 12,000/4,000개를 사용합니다.
5. 10개 그룹 모두를 포함하는 로컬 CPU 학습 설정입니다. CPU 속도에 따라 수십 분이 걸릴 수 있으며, 검증 지표를 보고 학습을 멈춥니다.

부품과 연결되지 않는 손상 bbox는 임의 라벨을 만들지 않고 제외합니다. crop 수는 클래스별 상한이며, 표본 내 실제 라벨 수가 부족하면 더 적게 생성됩니다.

In [1]:
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "train" / "part").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / "train" / "part" / "160. 차량파손 이미지 데이터" / "01.데이터"
LOCAL_RUN_ROOT = PROJECT_ROOT / "train" / "car_damage_runs" / "part_classifier_effnetb3"
TEST_MODE = True  # 빠른 테스트; 전체 데이터 학습은 False
RUN_NAME = f"effb3_damage_part_local_{datetime.now():%Y%m%d_%H%M%S}" if TEST_MODE else "effb3_damage_part_v1"
RESUME = False

EPOCHS = 5 if TEST_MODE else 10
EARLY_STOPPING_PATIENCE = 2 if TEST_MODE else 3
BATCH_SIZE = 4 if TEST_MODE else 16
IMAGE_SIZE = 224 if TEST_MODE else 300
NUM_WORKERS = 0 if TEST_MODE else 4
OVERLAP_THRESHOLD = 0.25
MAX_TRAIN_JSONS = 12000 if TEST_MODE else None
MAX_VAL_JSONS = 4000 if TEST_MODE else None
MAX_TRAIN_CROPS_PER_CLASS = 200 if TEST_MODE else None
MAX_VAL_CROPS_PER_CLASS = 50 if TEST_MODE else None
USE_PRETRAINED = True
CROP_ROOT = PROJECT_ROOT / "train" / "car_damage_runs" / ("effb3_damage_part_local_2k_crops" if TEST_MODE else "effb3_damage_part_crops")

ARCHIVES = {
    "train_images": DATA_ROOT / "1.Training/1.원천데이터/TS_damage_part.zip",
    "train_labels": DATA_ROOT / "1.Training/2.라벨링데이터/TL_damage_part.zip",
    "val_images": DATA_ROOT / "2.Validation/1.원천데이터/VS_damage_part.zip",
    "val_labels": DATA_ROOT / "2.Validation/2.라벨링데이터/VL_damage_part.zip",
}
RUN_DIR = LOCAL_RUN_ROOT / RUN_NAME
LAST_CHECKPOINT = RUN_DIR / "last.pt"
if RESUME and not LAST_CHECKPOINT.is_file():
    raise FileNotFoundError(f"이어 학습할 last.pt가 없습니다: {LAST_CHECKPOINT}")
if not RESUME and RUN_DIR.exists():
    raise FileExistsError(
        f"실행 폴더가 이미 있습니다: {RUN_DIR}. RESUME=True로 이어가거나 RUN_NAME을 바꾸세요."
    )

missing = [str(path) for path in ARCHIVES.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "로컬에서 데이터 ZIP을 찾을 수 없습니다. DATA_ROOT를 수정하세요:\n"
        + "\n".join(missing)
    )
for key, path in ARCHIVES.items():
    print(f"{key}: {path.stat().st_size / 1e9:.2f} GB — {path}")

train_images: 13.69 GB — c:\code\00_Car_damage\train\part\160. 차량파손 이미지 데이터\01.데이터\1.Training\1.원천데이터\TS_damage_part.zip
train_labels: 0.12 GB — c:\code\00_Car_damage\train\part\160. 차량파손 이미지 데이터\01.데이터\1.Training\2.라벨링데이터\TL_damage_part.zip
val_images: 2.19 GB — c:\code\00_Car_damage\train\part\160. 차량파손 이미지 데이터\01.데이터\2.Validation\1.원천데이터\VS_damage_part.zip
val_labels: 0.02 GB — c:\code\00_Car_damage\train\part\160. 차량파손 이미지 데이터\01.데이터\2.Validation\2.라벨링데이터\VL_damage_part.zip


In [2]:
import os
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
LOCAL_ARCHIVES = ARCHIVES
print(f"Device: {device} | PyTorch: {torch.__version__}")
for key, path in LOCAL_ARCHIVES.items():
    print(f"{key}: {path.stat().st_size / 1e9:.2f} GB — {path}")

Device: cpu | PyTorch: 2.13.0+cpu
train_images: 13.69 GB — c:\code\00_Car_damage\train\part\160. 차량파손 이미지 데이터\01.데이터\1.Training\1.원천데이터\TS_damage_part.zip
train_labels: 0.12 GB — c:\code\00_Car_damage\train\part\160. 차량파손 이미지 데이터\01.데이터\1.Training\2.라벨링데이터\TL_damage_part.zip
val_images: 2.19 GB — c:\code\00_Car_damage\train\part\160. 차량파손 이미지 데이터\01.데이터\2.Validation\1.원천데이터\VS_damage_part.zip
val_labels: 0.02 GB — c:\code\00_Car_damage\train\part\160. 차량파손 이미지 데이터\01.데이터\2.Validation\2.라벨링데이터\VL_damage_part.zip


In [3]:
import json
import random
import math
import zipfile
from collections import Counter, defaultdict

PART_TO_CLASS = {
    "Front bumper": "앞범퍼",
    "Front fender(L)": "앞펜더", "Front fender(R)": "앞펜더",
    "Head lights(L)": "전조등", "Head lights(R)": "전조등",
    "Rear bumper": "뒷범퍼",
    "Rear fender(L)": "뒷펜더", "Rear fender(R)": "뒷펜더",
    "Rear lamp(L)": "후미등", "Rear lamp(R)": "후미등",
    "Front door(L)": "옆면", "Front door(R)": "옆면",
    "Rear door(L)": "옆면", "Rear door(R)": "옆면",
    "Side mirror(L)": "사이드미러", "Side mirror(R)": "사이드미러",
    "Rocker panel(L)": "스텝", "Rocker panel(R)": "스텝",
    "Front Wheel(L)": "타이어/휠", "Front Wheel(R)": "타이어/휠",
    "Rear Wheel(L)": "타이어/휠", "Rear Wheel(R)": "타이어/휠",
}
CLASS_NAMES = ["앞범퍼", "앞펜더", "전조등", "뒷범퍼", "뒷펜더", "후미등", "옆면", "사이드미러", "스텝", "타이어/휠"]

def valid_bbox(value):
    if not isinstance(value, list) or len(value) < 4:
        return None
    try:
        x, y, width, height = (float(item) for item in value[:4])
    except (TypeError, ValueError):
        return None
    if not all(math.isfinite(item) for item in (x, y, width, height)) or width <= 0 or height <= 0:
        return None
    return x, y, width, height

def overlap_fraction(damage_box, part_box):
    x, y, width, height = damage_box
    px, py, pwidth, pheight = part_box
    intersection = max(0.0, min(x + width, px + pwidth) - max(x, px)) * max(
        0.0, min(y + height, py + pheight) - max(y, py)
    )
    return intersection / (width * height)

def build_damage_crop_records(image_archive_path, label_archive_path, max_jsons=None, seed=42):
    records_by_image = defaultdict(list)
    report = Counter()
    with zipfile.ZipFile(image_archive_path) as image_zip, zipfile.ZipFile(label_archive_path) as label_zip:
        image_members = {
            Path(name).stem.casefold(): name
            for name in image_zip.namelist()
            if Path(name).suffix.casefold() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        }
        label_names = [name for name in label_zip.namelist() if name.lower().endswith(".json")]
        report["label_jsons"] = len(label_names)
        if max_jsons is not None and len(label_names) > max_jsons:
            label_names = random.Random(seed).sample(label_names, max_jsons)
            report["sampled_label_jsons"] = len(label_names)
        for json_index, label_name in enumerate(label_names, start=1):
            if json_index % 1000 == 0:
                print(f"라벨 읽기 {label_archive_path.name}: {json_index:,}/{len(label_names):,}")
            stem = Path(label_name).stem.casefold()
            image_member = image_members.get(stem)
            if image_member is None:
                report["json_without_image"] += 1
                continue
            try:
                annotation_data = json.loads(label_zip.read(label_name).decode("utf-8-sig"))
            except (UnicodeError, json.JSONDecodeError) as error:
                raise ValueError(f"Invalid annotation JSON {label_name}: {error}") from error
            annotations = annotation_data.get("annotations", [])
            if not isinstance(annotations, list):
                report["invalid_annotations_list"] += 1
                continue
            parts = []
            damages = []
            for annotation_index, annotation in enumerate(annotations):
                if not isinstance(annotation, dict):
                    continue
                box = valid_bbox(annotation.get("bbox"))
                if box is None:
                    continue
                part_name = annotation.get("part")
                damage_name = annotation.get("damage")
                if isinstance(part_name, str) and part_name.strip():
                    mapped_part = PART_TO_CLASS.get(part_name.strip())
                    if mapped_part is not None:
                        parts.append((mapped_part, box))
                if isinstance(damage_name, str) and damage_name.strip():
                    damages.append((annotation_index, box))
            for annotation_index, damage_box in damages:
                report["damage_objects"] += 1
                candidates = [
                    (overlap_fraction(damage_box, part_box), part_name)
                    for part_name, part_box in parts
                ]
                if not candidates:
                    report["damage_without_part_annotation"] += 1
                    continue
                best_overlap, best_part = max(candidates, key=lambda item: item[0])
                if best_overlap < OVERLAP_THRESHOLD:
                    report["damage_below_overlap_threshold"] += 1
                    continue
                report["assigned_crops"] += 1
                report[f"class::{best_part}"] += 1
                records_by_image[image_member].append(
                    (annotation_index, damage_box, best_part, best_overlap)
                )
    return records_by_image, report

train_records, train_report = build_damage_crop_records(
    LOCAL_ARCHIVES["train_images"], LOCAL_ARCHIVES["train_labels"], MAX_TRAIN_JSONS, seed=42
)
val_records, val_report = build_damage_crop_records(
    LOCAL_ARCHIVES["val_images"], LOCAL_ARCHIVES["val_labels"], MAX_VAL_JSONS, seed=43
)
train_class_counts = {key.removeprefix("class::"): value for key, value in train_report.items() if key.startswith("class::")}
val_class_counts = {key.removeprefix("class::"): value for key, value in val_report.items() if key.startswith("class::")}
shared_classes = set(train_class_counts) & set(val_class_counts)
class_names = [name for name in CLASS_NAMES if name in shared_classes]
missing_classes = sorted(set(CLASS_NAMES) - set(class_names))
if missing_classes and not TEST_MODE:
    raise ValueError(f"Train/validation에 crop이 없는 부품 그룹이 있습니다: {missing_classes}")
if len(class_names) < 2:
    raise ValueError(f"분류에 필요한 공통 part 클래스가 부족합니다: {class_names}")
class_name_set = set(class_names)
train_records = {image: [record for record in records if record[2] in class_name_set] for image, records in train_records.items()}
val_records = {image: [record for record in records if record[2] in class_name_set] for image, records in val_records.items()}
train_records = {image: records for image, records in train_records.items() if records}
val_records = {image: records for image, records in val_records.items() if records}
print(f"테스트 모드: {TEST_MODE} | 선택한 JSON: train {train_report.get('sampled_label_jsons', train_report['label_jsons']):,}, val {val_report.get('sampled_label_jsons', val_report['label_jsons']):,}")
print(f"사용할 part 클래스 {len(class_names)}/{len(CLASS_NAMES)}개: {class_names}")
if missing_classes:
    print(f"샘플에 없어 이번 테스트에서 제외한 클래스: {missing_classes}")
print(f"Train 손상 객체 {train_report['damage_objects']:,}, 후보 crop {train_report['assigned_crops']:,}")
print(f"Validation 손상 객체 {val_report['damage_objects']:,}, 후보 crop {val_report['assigned_crops']:,}")
print("Train crop 수 상위 클래스:", sorted(
        ((key.removeprefix("class::"), value) for key, value in train_report.items() if key.startswith("class::")),
        key=lambda item: item[1], reverse=True,
    )[:12])

테스트 모드: True | 선택한 JSON: train 100, val 50
사용할 part 클래스 4개: ['Front bumper', 'Rear bumper', 'Front fender(L)', 'Bonnet']
Train 손상 객체 356, 후보 crop 327
Validation 손상 객체 202, 후보 crop 187
Train crop 수 상위 클래스: [('Front bumper', 111), ('Rear bumper', 76), ('Front fender(R)', 33), ('Front fender(L)', 12), ('Rear door(R)', 12), ('Rear fender(R)', 11), ('Rear fender(L)', 11), ('Head lights(R)', 9), ('Bonnet', 8), ('Front Wheel(R)', 6), ('Front door(L)', 6), ('Front Wheel(L)', 5)]


In [4]:
import io
import shutil
from PIL import Image

CROP_MANIFEST = CROP_ROOT / "dataset_manifest.json"
class_to_dir = {name: f"{index:02d}_{name.replace('/', '-')}" for index, name in enumerate(class_names)}
directory_to_class = {directory: name for name, directory in class_to_dir.items()}
if CROP_MANIFEST.is_file():
    manifest = json.loads(CROP_MANIFEST.read_text(encoding="utf-8"))
    expected_settings = {
        "class_names": class_names,
        "test_mode": TEST_MODE,
        "max_train_jsons": MAX_TRAIN_JSONS,
        "max_val_jsons": MAX_VAL_JSONS,
        "max_train_crops_per_class": MAX_TRAIN_CROPS_PER_CLASS,
        "max_val_crops_per_class": MAX_VAL_CROPS_PER_CLASS,
        "overlap_threshold": OVERLAP_THRESHOLD,
    }
    mismatched_settings = [key for key, value in expected_settings.items() if manifest.get(key) != value]
    if mismatched_settings:
        raise ValueError(
            "기존 crop 데이터의 설정이 현재와 다릅니다 "
            f"({', '.join(mismatched_settings)}). CROP_ROOT를 새 경로로 바꾸세요."
        )
    print(f"기존 crop 데이터 재사용: {CROP_ROOT}")
elif CROP_ROOT.exists() and any(CROP_ROOT.iterdir()):
    raise FileExistsError(
        f"불완전하거나 다른 데이터가 있는 폴더입니다: {CROP_ROOT}. "
        "데이터를 보존하기 위해 자동 삭제하지 않았습니다. CROP_ROOT를 새 경로로 바꾸세요."
    )
else:
    CROP_ROOT.mkdir(parents=True, exist_ok=True)
    crop_totals = {}
    for split, records_by_image, archive_path in (
        ("train", train_records, LOCAL_ARCHIVES["train_images"]),
        ("val", val_records, LOCAL_ARCHIVES["val_images"]),
    ):
        for directory in class_to_dir.values():
            (CROP_ROOT / split / directory).mkdir(parents=True, exist_ok=True)
        split_counts = Counter()
        crop_limit = MAX_TRAIN_CROPS_PER_CLASS if split == "train" else MAX_VAL_CROPS_PER_CLASS
        image_count = len(records_by_image)
        image_items = list(records_by_image.items())
        random.Random(42 if split == "train" else 43).shuffle(image_items)
        with zipfile.ZipFile(archive_path) as image_zip:
            for image_index, (image_member, image_records) in enumerate(image_items, start=1):
                with Image.open(io.BytesIO(image_zip.read(image_member))) as source_image:
                    source_image = source_image.convert("RGB")
                    image_width, image_height = source_image.size
                    for annotation_index, (x, y, width, height), part_name, overlap in image_records:
                        if crop_limit is not None and split_counts[part_name] >= crop_limit:
                            continue
                        center_x, center_y = x + width / 2, y + height / 2
                        crop_width = max(width * 1.5, 48)
                        crop_height = max(height * 1.5, 48)
                        left = max(0, math.floor(center_x - crop_width / 2))
                        top = max(0, math.floor(center_y - crop_height / 2))
                        right = min(image_width, math.ceil(center_x + crop_width / 2))
                        bottom = min(image_height, math.ceil(center_y + crop_height / 2))
                        if right <= left or bottom <= top:
                            split_counts["invalid_crop"] += 1
                            continue
                        crop = source_image.crop((left, top, right, bottom))
                        output_name = f"{Path(image_member).stem}__ann{annotation_index:04d}.jpg"
                        output_path = CROP_ROOT / split / class_to_dir[part_name] / output_name
                        crop.save(output_path, format="JPEG", quality=90)
                        split_counts[part_name] += 1
                if crop_limit is not None and all(split_counts[name] >= crop_limit for name in class_names):
                    break
                if image_index % 2000 == 0 or image_index == image_count:
                    print(f"{split}: {image_index:,}/{image_count:,} source images — {sum(split_counts.values()):,} crops")
        print(f"{split} crop counts:", dict(split_counts))
        crop_totals[split] = sum(split_counts[name] for name in class_names)
    manifest = {
        "class_names": class_names,
        "train_crops": crop_totals["train"],
        "val_crops": crop_totals["val"],
        "test_mode": TEST_MODE,
        "max_train_jsons": MAX_TRAIN_JSONS,
        "max_val_jsons": MAX_VAL_JSONS,
        "max_train_crops_per_class": MAX_TRAIN_CROPS_PER_CLASS,
        "max_val_crops_per_class": MAX_VAL_CROPS_PER_CLASS,
        "overlap_threshold": OVERLAP_THRESHOLD,
        "train_label_report": dict(train_report),
        "val_label_report": dict(val_report),
    }
    CROP_MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Crop 데이터와 manifest 생성 완료: {CROP_ROOT}")

기존 crop 데이터 재사용: c:\code\00_Car_damage\train\car_damage_runs\effb3_damage_part_local_smoke_crops


In [5]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

if not CROP_MANIFEST.is_file():
    raise FileNotFoundError(f"Crop manifest가 없습니다: {CROP_MANIFEST}")

weights = models.EfficientNet_B3_Weights.DEFAULT if USE_PRETRAINED else None
mean, std = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    # 좌우 반전은 L/R 부품 라벨을 바꾸므로 적용하지 않습니다.
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
val_transform = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
train_dataset = datasets.ImageFolder(CROP_ROOT / "train", transform=train_transform)
val_dataset = datasets.ImageFolder(CROP_ROOT / "val", transform=val_transform)
if train_dataset.classes != val_dataset.classes:
    raise ValueError("Train/validation part 클래스 폴더가 일치하지 않습니다.")
if train_dataset.class_to_idx != val_dataset.class_to_idx:
    raise ValueError("Train/validation class index가 일치하지 않습니다.")
part_names = [directory_to_class[name] for name in train_dataset.classes]
train_counts = torch.bincount(torch.tensor(train_dataset.targets), minlength=len(part_names)).float()
val_counts = torch.bincount(torch.tensor(val_dataset.targets), minlength=len(part_names))
if torch.any(train_counts == 0) or torch.any(val_counts == 0):
    raise ValueError("각 part 클래스에 train/validation crop이 모두 있어야 합니다.")

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda", persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda", persistent_workers=NUM_WORKERS > 0,
)

print(f"Part classes ({len(part_names)}): {part_names}")
print(f"Train crops: {len(train_dataset):,} | Validation crops: {len(val_dataset):,}")
print("Train class counts:", {name: int(train_counts[i]) for i, name in enumerate(part_names)})

Part classes (4): ['Front bumper', 'Rear bumper', 'Front fender(L)', 'Bonnet']
Train crops: 20 | Validation crops: 12
Train class counts: {'Front bumper': 5, 'Rear bumper': 5, 'Front fender(L)': 5, 'Bonnet': 5}


In [6]:
import csv
import time
from sklearn.metrics import accuracy_score, f1_score, classification_report

RUN_DIR = LOCAL_RUN_ROOT / RUN_NAME
LAST_CHECKPOINT = RUN_DIR / "last.pt"
BEST_CHECKPOINT = RUN_DIR / "best.pt"
HISTORY_CSV = RUN_DIR / "history.csv"
model = models.efficientnet_b3(weights=weights)
feature_count = model.classifier[1].in_features
model.classifier[1] = nn.Linear(feature_count, len(part_names))
model = model.to(device)

class_weights = train_counts.clamp_min(1).reciprocal().sqrt()
class_weights = class_weights / class_weights.mean()
class_weights = class_weights.clamp(max=5.0).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=1,
)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
start_epoch = 0
best_macro_f1 = 0.0
epochs_without_improvement = 0

if RESUME:
    if not LAST_CHECKPOINT.is_file():
        raise FileNotFoundError(f"이어 학습할 last.pt가 없습니다: {LAST_CHECKPOINT}")
    checkpoint = torch.load(LAST_CHECKPOINT, map_location=device, weights_only=False)
    if checkpoint.get("class_names") != part_names:
        raise ValueError("체크포인트의 클래스와 현재 데이터 클래스가 다릅니다.")
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    scaler.load_state_dict(checkpoint["scaler"])
    start_epoch = checkpoint["epoch"]
    best_macro_f1 = checkpoint["best_macro_f1"]
    epochs_without_improvement = checkpoint.get("epochs_without_improvement", 0)
    print(f"Epoch {start_epoch}까지 저장된 체크포인트에서 이어갑니다.")
elif RUN_DIR.exists():
    raise FileExistsError(
        f"실행 폴더가 이미 있습니다: {RUN_DIR}. RESUME=True로 이어가거나 RUN_NAME을 바꾸세요."
    )

RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"EfficientNet-B3 | classes={len(part_names)} | device={device} | run={RUN_DIR}")

EfficientNet-B3 | classes=4 | device=cpu | run=c:\code\00_Car_damage\train\car_damage_runs\part_classifier_effnetb3\effb3_damage_part_local_smoke_20261101_122357


In [7]:
from tqdm.auto import tqdm

def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    all_targets, all_predictions = [], []
    progress_bar = tqdm(loader, leave=False, desc="train" if training else "validation")
    for images, targets in progress_bar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, targets)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        total_loss += loss.item() * targets.size(0)
        predictions = logits.argmax(dim=1)
        all_targets.extend(targets.cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())
        progress_bar.set_postfix(loss=f"{loss.item():.3f}")
    average_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(all_targets, all_predictions)
    macro_f1 = f1_score(all_targets, all_predictions, average="macro", zero_division=0)
    return average_loss, accuracy, macro_f1, all_targets, all_predictions

for epoch in range(start_epoch, EPOCHS):
    epoch_started = time.monotonic()
    train_loss, train_accuracy, train_f1, _, _ = run_epoch(train_loader, training=True)
    val_loss, val_accuracy, val_f1, val_targets, val_predictions = run_epoch(val_loader, training=False)
    scheduler.step(val_f1)
    improved = val_f1 > best_macro_f1
    if improved:
        best_macro_f1 = val_f1
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    epoch_number = epoch + 1
    row = {
        "epoch": epoch_number,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "train_macro_f1": train_f1,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_f1,
        "seconds": time.monotonic() - epoch_started,
    }
    print(
        f"Epoch {epoch_number}/{EPOCHS} | "
        f"train loss {train_loss:.4f}, acc {train_accuracy:.4f}, macro-F1 {train_f1:.4f} | "
        f"val loss {val_loss:.4f}, acc {val_accuracy:.4f}, macro-F1 {val_f1:.4f} | "
        f"{row['seconds'] / 60:.1f} min"
    )
    with HISTORY_CSV.open("a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=row.keys())
        if file.tell() == 0:
            writer.writeheader()
        writer.writerow(row)

    checkpoint = {
        "epoch": epoch_number,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "class_names": part_names,
        "best_macro_f1": best_macro_f1,
        "epochs_without_improvement": epochs_without_improvement,
        "image_size": IMAGE_SIZE,
        "architecture": "efficientnet_b3",
    }
    temporary_checkpoint = RUN_DIR / "last.tmp.pt"
    torch.save(checkpoint, temporary_checkpoint)
    temporary_checkpoint.replace(LAST_CHECKPOINT)
    if improved or not BEST_CHECKPOINT.exists():
        temporary_best = RUN_DIR / "best.tmp.pt"
        torch.save(checkpoint, temporary_best)
        temporary_best.replace(BEST_CHECKPOINT)
    print(f"체크포인트 저장: {LAST_CHECKPOINT}")
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Validation Macro F1이 {EARLY_STOPPING_PATIENCE} epoch 동안 개선되지 않아 조기 종료합니다.")
        break

print("학습 완료")
print(classification_report(val_targets, val_predictions, target_names=part_names, zero_division=0))

c:\Users\hwi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch 1/1 | train loss 1.4421, acc 0.1000, macro-F1 0.0667 | val loss 1.3901, acc 0.2500, macro-F1 0.1000 | 0.1 min
체크포인트 저장: c:\code\00_Car_damage\train\car_damage_runs\part_classifier_effnetb3\effb3_damage_part_local_smoke_20261101_122357\last.pt
학습 완료
                 precision    recall  f1-score   support

   Front bumper       0.00      0.00      0.00         3
    Rear bumper       0.25      1.00      0.40         3
Front fender(L)       0.00      0.00      0.00         3
         Bonnet       0.00      0.00      0.00         3

       accuracy                           0.25        12
      macro avg       0.06      0.25      0.10        12
   weighted avg       0.06      0.25      0.10        12



In [8]:
print(f"Best checkpoint: {BEST_CHECKPOINT} — exists={BEST_CHECKPOINT.is_file()}")
print(f"Last checkpoint: {LAST_CHECKPOINT} — exists={LAST_CHECKPOINT.is_file()}")
print(f"Training history: {HISTORY_CSV} — exists={HISTORY_CSV.is_file()}")
print(f"Part crop dataset: {CROP_ROOT}")

Best checkpoint: c:\code\00_Car_damage\train\car_damage_runs\part_classifier_effnetb3\effb3_damage_part_local_smoke_20261101_122357\best.pt — exists=True
Last checkpoint: c:\code\00_Car_damage\train\car_damage_runs\part_classifier_effnetb3\effb3_damage_part_local_smoke_20261101_122357\last.pt — exists=True
Training history: c:\code\00_Car_damage\train\car_damage_runs\part_classifier_effnetb3\effb3_damage_part_local_smoke_20261101_122357\history.csv — exists=True
Part crop dataset: c:\code\00_Car_damage\train\car_damage_runs\effb3_damage_part_local_smoke_crops
